In [92]:
import enum
import re
from collections import Counter
from dataclasses import dataclass
from typing import Callable

import polars as pl
from cachetools.func import lru_cache



In [93]:
@dataclass
class Data:
    questions: pl.DataFrame
    answers: pl.DataFrame
    experiments: pl.DataFrame
    runs: pl.DataFrame


def load_tables():
    return Data(
        questions=pl.read_parquet("results/questions.parquet"),
        answers=pl.read_parquet("results/answers.parquet"),
        experiments=pl.read_parquet("results/experiments.parquet"),
        runs=pl.read_parquet("results/runs.parquet"),
    )


def build_analysis_frame():
    data = load_tables()

    df = (
        data.answers.join(data.runs, on="run_id", how="left")
        .join(data.experiments, on="experiment_id", how="left")
        .join(data.questions, on="question_id", how="left")
    )
    return df

In [94]:
data = load_tables()
assert not data.questions.drop("question_id").is_duplicated().any()

In [95]:
df = build_analysis_frame()
df.head()

shape: (5, 22)
┌─────┬────────┬─────────────┬─────────────┬───┬────────────┬────────────┬────────────┬────────────┐
│ id  ┆ run_id ┆ message_ids ┆ question_id ┆ … ┆ question   ┆ answer_opt ┆ answer_ind ┆ answer_str │
│ --- ┆ ---    ┆ ---         ┆ ---         ┆   ┆ ---        ┆ ions       ┆ ex         ┆ ing        │
│ i64 ┆ i64    ┆ list[i64]   ┆ i64         ┆   ┆ str        ┆ ---        ┆ ---        ┆ ---        │
│     ┆        ┆             ┆             ┆   ┆            ┆ list[str]  ┆ i64        ┆ str        │
╞═════╪════════╪═════════════╪═════════════╪═══╪════════════╪════════════╪════════════╪════════════╡
│ 0   ┆ 0      ┆ []          ┆ 0           ┆ … ┆ Q: A store ┆ ["3 to 1", ┆ 3          ┆ D          │
│     ┆        ┆             ┆             ┆   ┆ sells two  ┆ "4 to 3",  ┆            ┆            │
│     ┆        ┆             ┆             ┆   ┆ items for… ┆ … "5 to 3… ┆            ┆            │
│ 1   ┆ 0      ┆ []          ┆ 1           ┆ … ┆ Q: 00      ┆ ["360      ┆ 0          ┆ A          │
│     ┆        ┆             ┆             ┆   ┆ p.m.,      ┆ mph", "450 ┆            ┆            │
│     ┆        ┆             ┆             ┆   ┆ traveled   ┆ mph", …    ┆            ┆            │
│     ┆        ┆             ┆             ┆   ┆ 540 miles… ┆ "540 …     ┆            ┆            │
│ 2   ┆ 0      ┆ []          ┆ 2           ┆ … ┆ Q: A       ┆ ["Y = 18 - ┆ 0          ┆ A          │
│     ┆        ┆             ┆             ┆   ┆ research   ┆ 0.05 X, r  ┆            ┆            │
│     ┆        ┆             ┆             ┆   ┆ worker was ┆ = - .4867… ┆            ┆            │
│     ┆        ┆             ┆             ┆   ┆ inter…     ┆            ┆            ┆            │
│ 3   ┆ 0      ┆ []          ┆ 3           ┆ … ┆ Q: Solve   ┆ ["145      ┆ 7          ┆ H          │
│     ┆        ┆             ┆             ┆   ┆ the        ┆ plots, 45  ┆            ┆            │
│     ┆        ┆             ┆             ┆   ┆ following  ┆ graduating ┆            ┆            │
│     ┆        ┆             ┆             ┆   ┆ problem…   ┆ sen…       ┆            ┆            │
│ 4   ┆ 0      ┆ []          ┆ 4           ┆ … ┆ Q: Joe's   ┆ ["$48",    ┆ 7          ┆ H          │
│     ┆        ┆             ┆             ┆   ┆ Department ┆ "$58", …   ┆            ┆            │
│     ┆        ┆             ┆             ┆   ┆ Store      ┆ "$70"]     ┆            ┆            │
│     ┆        ┆             ┆             ┆   ┆ wish…      ┆            ┆            ┆            │
└─────┴────────┴─────────────┴─────────────┴───┴────────────┴────────────┴────────────┴────────────┘

## Parsing Answers

In [96]:
INPUT_WAS_EMPTY = " --- empty input --- "
NOT_PARSABLE = "N/A"

INVALID_PARSE_VALUES = (INPUT_WAS_EMPTY, NOT_PARSABLE)

ALL_VOTES_INVALID = " --- all votes invalid --- "
DIFFERENT_VOTES = " --- votes different --- "

In [97]:
OPTION_LETTERS = ["A", "B", "C", "D", "E", "F", "G", "H", "I", "J"]

_ANSWER_PATTERNS = [
    r"answer\s+is\s*:?\s*\(?([A-J])\)?",  # ...answer is (C)... | ...answer is C ...
    r"^\s*([A-J])\s*$",  # single letter: C
    r"^\s*\(?\s*([A-J])\s*\)?\s*$",  # single letter in braces: (C)
    r"answer\s*:\s*\(?\s*([A-J])\s*\)?\s*[\.\!\?]*\s*$",
    # ... Answer: (C) | ... Answer: C  |# at the end of the string with optional punctuation
    r"^\s*\(\s*([A-J])\s*\)\s*:?.*$",  # (C): ... | (C) .... |# at the beginning of the string
]


@lru_cache
def parse_answer(answer: str | None) -> str:
    if answer is None:
        return INPUT_WAS_EMPTY

    answer = answer.replace("*", "")

    for pat in _ANSWER_PATTERNS:
        m = re.search(pat, answer, flags=re.IGNORECASE)
        if m and ((final_letter := m.group(1).upper()) in OPTION_LETTERS):
            return final_letter

    return NOT_PARSABLE


class VotingStrategy(enum.Enum):
    SINGULARITY = "singularity"  # all have to vote the same, otherwise invalid
    MAJORITY = "majority"  # the majority vote >50%

    def parse(self, votes: list[str], use_none_for_undecided: bool = False) -> str:
        if len(votes) == 0:
            return ALL_VOTES_INVALID if not use_none_for_undecided else None
        elif len(votes) == 1:
            return votes[0]
        elif len(set(votes)) == 1:
            return votes[0]

        match self:
            case VotingStrategy.SINGULARITY:
                return DIFFERENT_VOTES if not use_none_for_undecided else None
            case VotingStrategy.MAJORITY:
                most_common, second_most_common = Counter(votes).most_common(2)
                if most_common[1] == second_most_common[1]:
                    return DIFFERENT_VOTES if not use_none_for_undecided else None
                else:
                    return most_common[0]

            case _:
                raise NotImplementedError()


def create_group_answer_with(
        strategy: VotingStrategy, use_none_for_undecided: bool = False
) -> Callable[[list[str]], str]:
    def f(answers: list[str]) -> str:
        votes = [
            vote
            for a in answers
            if (vote := parse_answer(a)) not in INVALID_PARSE_VALUES
        ]
        return strategy.parse(votes, use_none_for_undecided)

    return f

In [98]:
# either answers_at_end of final_answer must be given
assert df.select(
    (
            (pl.col("answers_at_end").list.len() == 0) ^ (pl.col("final_answer").is_null())
    ).all()
).item()

df_with_parsed_answers = (
    df.with_columns(
        pl.col("final_answer").fill_null(
            pl.col("answers_at_end").map_elements(
                create_group_answer_with(VotingStrategy.MAJORITY)
            )
        )
    )
    .with_columns(
        given_answer=pl.col("final_answer").map_elements(
            parse_answer, return_dtype=pl.String
        )
    )
    .with_columns(pl.col("given_answer").replace(NOT_PARSABLE, None))
    .with_columns(is_correct=pl.col("given_answer").eq(pl.col("answer_string")))
)

assert df_with_parsed_answers["final_answer"].null_count() == 0
assert (
    df_with_parsed_answers["given_answer"]
    .drop_nulls()
    .is_in(OPTION_LETTERS, nulls_equal=True)
    .all()
)
assert (
    df_with_parsed_answers["answer_string"]
    .is_in(OPTION_LETTERS, nulls_equal=True)
    .all()
)

In [99]:
df_with_parsed_answers.select(
    ["answer_string", "answers_at_end", "final_answer", "given_answer"]
)

shape: (24_064, 4)
┌───────────────┬─────────────────────────────────┬─────────────────────────────────┬──────────────┐
│ answer_string ┆ answers_at_end                  ┆ final_answer                    ┆ given_answer │
│ ---           ┆ ---                             ┆ ---                             ┆ ---          │
│ str           ┆ list[str]                       ┆ str                             ┆ str          │
╞═══════════════╪═════════════════════════════════╪═════════════════════════════════╪══════════════╡
│ D             ┆ ["(F): 1 to 3", "(Single answe… ┆  --- votes different ---        ┆ null         │
│ A             ┆ ["I understand that I have tak… ┆  --- votes different ---        ┆ null         │
│ A             ┆ ["(I)'s explanation seems inco… ┆ F                               ┆ F            │
│ H             ┆ ["(60 plots, 45 graduating sen… ┆  --- votes different ---        ┆ null         │
│ H             ┆ ["Sure, here's the solution:    ┆  --- votes different ---        ┆ null         │
│               ┆                                 ┆                                 ┆              │
│               ┆ …                               ┆                                 ┆              │
│ …             ┆ …                               ┆ …                               ┆ …            │
│ B             ┆ []                              ┆ To determine the length of the… ┆ F            │
│ D             ┆ []                              ┆ Let's think through it step-by… ┆ E            │
│ A             ┆ []                              ┆ To find the average flow veloc… ┆ null         │
│ F             ┆ []                              ┆ To find the percentage modulat… ┆ A            │
│ I             ┆ []                              ┆ To compute the local friction … ┆ null         │
└───────────────┴─────────────────────────────────┴─────────────────────────────────┴──────────────┘

# Direct Experiment Comparison

In [100]:
p = 1 / len(OPTION_LETTERS)

(
    df_with_parsed_answers.group_by("run_identifier").agg(
        accuracy_ignore_nulls=pl.col("is_correct").mean(),
        accuracy_null_false=pl.col("is_correct").fill_null(False).mean(),
        accuracy_probabilistic=(
            pl.when(pl.col("is_correct").is_null())
            .then(p)
            .otherwise(pl.col("is_correct").cast(pl.Float64))
        ).mean(),
        accuracy_random_baseline=p,
    )
)

# TODO: Add used token overview

run_identifier,accuracy_ignore_nulls,accuracy_null_false,accuracy_probabilistic,accuracy_random_baseline
str,f64,f64,f64,f64
"""standard - 2026-01-08-09-49-55""",0.136291,0.106217,0.128283,0.1
"""multi-agent-debate - 2026-01-0…",0.107527,0.062334,0.104363,0.1


## Answer Distributions

In [101]:
q1 = (
    data.questions.group_by("answer_string")
    .len("amount")
    .rename({"answer_string": "letter"})
    .vstack(
        pl.DataFrame(
            {"letter": None, "amount": 0},
            orient="row",
            schema=pl.Schema({"letter": pl.String, "amount": pl.UInt32}),
        )
    )
    .with_columns(meaning=pl.lit("Amount of Questions with this answer letter."))
)

q2 = (
    df_with_parsed_answers.group_by("run_identifier", "given_answer")
    .agg(amount=pl.col("given_answer").len())
    .with_columns(
        meaning=pl.lit("Answers with this letter in ") + pl.col("run_identifier")
    )
    .rename({"given_answer": "letter"})
    .drop("run_identifier")
)

letter_dist = q1.vstack(q2)

In [102]:
letter_dist.plot.bar(
    x="letter",
    y="amount",
    color="meaning",
    xOffset="meaning",
).configure_legend(labelLimit=400)

alt.Chart(...)

In [103]:
letter_dist.plot.bar(
    x="letter",
    y="amount",
    column="meaning",
)

alt.Chart(...)

## Group Decision Scheme Analysis

In [104]:
def count_equal_elements(
        df: pl.DataFrame, list_col: str, compare_col: str, *, new_col: str
):
    CORRECT = "______custom________correct_column"
    ROW_IDX = "______custom________row_index"
    return (
        df.with_row_index(ROW_IDX)
        .explode(list_col)
        .with_columns((pl.col(list_col) == pl.col(compare_col)).alias(CORRECT))
        .group_by(ROW_IDX, maintain_order=True)
        .agg(pl.col(CORRECT).sum().alias(new_col))
        .join(df.with_row_index(ROW_IDX), on=ROW_IDX, how="left")
        .drop(ROW_IDX)
    )


def count_unequal_elements(
        df: pl.DataFrame, list_col: str, compare_col: str, *, new_col: str
):
    CORRECT = "______custom________correct_column"
    ROW_IDX = "______custom________row_index"
    return (
        df.with_row_index(ROW_IDX)
        .explode(list_col)
        .with_columns((pl.col(list_col) != pl.col(compare_col)).alias(CORRECT))
        .group_by(ROW_IDX, maintain_order=True)
        .agg(pl.col(CORRECT).sum().alias(new_col))
        .join(df.with_row_index(ROW_IDX), on=ROW_IDX, how="left")
        .drop(ROW_IDX)
    )

In [105]:
xxx = (
    df_with_parsed_answers.filter(
        (pl.col("answers_at_end").list.len() != 0)
        & (pl.col("answers_at_beginning").list.len() != 0)
    )
    .with_columns(
        pl.col("answers_at_beginning")
        .list.eval(pl.element().map_elements(parse_answer, return_dtype=pl.String))
        .alias("parsed_answers_at_beginning"),
        pl.col("answers_at_end")
        .list.eval(pl.element().map_elements(parse_answer, return_dtype=pl.String))
        .alias("parsed_answers_at_end"),
    )
    .select(
        [
            "parsed_answers_at_beginning",
            "parsed_answers_at_end",
            "answer_string",
            "run_identifier",
        ]
    )
    .with_columns(
        pl.col("parsed_answers_at_beginning")
        .map_elements(
            create_group_answer_with(
                VotingStrategy.MAJORITY, use_none_for_undecided=True
            ),
            return_dtype=pl.String,
        )
        .alias("beginning_group_vote"),
        pl.col("parsed_answers_at_end")
        .map_elements(
            create_group_answer_with(
                VotingStrategy.MAJORITY, use_none_for_undecided=True
            ),
            return_dtype=pl.String,
        )
        .alias("end_group_vote"),
    )
    .with_columns(
        (pl.col("beginning_group_vote") == pl.col("answer_string")).alias(
            "beginning_group_correct"
        ),
        (pl.col("end_group_vote") == pl.col("answer_string")).alias(
            "end_group_correct"
        ),
    )
)
xxx = count_equal_elements(
    xxx,
    "parsed_answers_at_beginning",
    "answer_string",
    new_col="members_correct_beginning",
)
xxx = count_equal_elements(
    xxx, "parsed_answers_at_end", "answer_string", new_col="members_correct_end"
)
xxx = count_unequal_elements(
    xxx,
    "parsed_answers_at_beginning",
    "answer_string",
    new_col="members_incorrect_beginning",
)
xxx = count_unequal_elements(
    xxx, "parsed_answers_at_end", "answer_string", new_col="members_incorrect_end"
)

# TODO: for each strategy
# TODO: for ignoring null, treating null as wrong, treating null as random

assert (
    (xxx["members_correct_beginning"] + xxx["members_incorrect_beginning"]).eq(3)
).all()
assert ((xxx["members_correct_end"] + xxx["members_incorrect_end"]).eq(3)).all()

xxx

members_incorrect_end,members_incorrect_beginning,members_correct_end,members_correct_beginning,parsed_answers_at_beginning,parsed_answers_at_end,answer_string,run_identifier,beginning_group_vote,end_group_vote,beginning_group_correct,end_group_correct
u32,u32,u32,u32,list[str],list[str],str,str,str,str,bool,bool
3,3,0,0,"[""N/A"", ""N/A"", ""A""]","[""F"", ""N/A"", ""B""]","""D""","""multi-agent-debate - 2026-01-0…","""A""",null,false,null
2,2,1,1,"[""B"", ""B"", ""A""]","[""F"", ""B"", ""A""]","""A""","""multi-agent-debate - 2026-01-0…","""B""",null,false,null
3,2,0,1,"[""G"", ""A"", ""F""]","[""N/A"", ""F"", ""F""]","""A""","""multi-agent-debate - 2026-01-0…",null,"""F""",null,false
3,3,0,0,"[""F"", ""N/A"", ""N/A""]","[""N/A"", ""F"", ""I""]","""H""","""multi-agent-debate - 2026-01-0…","""F""",null,false,null
2,3,1,0,"[""N/A"", ""N/A"", ""J""]","[""C"", ""H"", ""N/A""]","""H""","""multi-agent-debate - 2026-01-0…","""J""",null,false,null
…,…,…,…,…,…,…,…,…,…,…,…
3,3,0,0,"[""N/A"", ""N/A"", ""N/A""]","[""A"", ""N/A"", ""N/A""]","""J""","""multi-agent-debate - 2026-01-0…",null,"""A""",null,false
2,2,1,1,"[""A"", ""I"", ""C""]","[""C"", ""N/A"", ""I""]","""I""","""multi-agent-debate - 2026-01-0…",null,null,null,null
3,3,0,0,"[""A"", ""N/A"", ""N/A""]","[""I"", ""E"", ""N/A""]","""F""","""multi-agent-debate - 2026-01-0…","""A""",null,false,null


In [113]:
(
    xxx.select(
        [
            "run_identifier",
            "end_group_correct",
            "members_correct_beginning",
            "members_incorrect_beginning",
            "beginning_group_correct",
        ]
    )
    .with_columns(
        # TODO: this can also be changed to ignore null or assume random choice then
        pl.col("beginning_group_correct").fill_null(False),
        pl.col("end_group_correct").fill_null(False),
    )
    .group_by(
        "run_identifier", "members_correct_beginning", "members_incorrect_beginning"
    )
    .agg(
        group_correct_end=pl.col("end_group_correct").mean(),
        group_incorrect_end=pl.lit(1).sub(pl.col("end_group_correct").mean()),
        group_correct_beginning=pl.col("beginning_group_correct").mean(),
        group_incorrect_beginning=pl.lit(1).sub(pl.col("beginning_group_correct").mean()),
    )
    .sort(["run_identifier", "members_correct_beginning"], descending=True)
)

run_identifier,members_correct_beginning,members_incorrect_beginning,group_correct_end,group_incorrect_end,group_correct_beginning,group_incorrect_beginning
str,u32,u32,f64,f64,f64,f64
"""multi-agent-debate - 2026-01-0…",3,0,0.368421,0.631579,1.0,0.0
"""multi-agent-debate - 2026-01-0…",2,1,0.255708,0.744292,1.0,0.0
"""multi-agent-debate - 2026-01-0…",1,2,0.147574,0.852426,0.201601,0.798399
"""multi-agent-debate - 2026-01-0…",0,3,0.04002,0.95998,0.0,1.0
